# [3장 통합 실습] 페이지를 모아 수집 파일 만들기

아래 문제 설명에 해당하는 완성 코드가 들어 있습니다. 실행하고 자신의 풀이와 비교하세요.

## 만들 것

Private LLM 도우미에 게시글을 연결할 때 첫 페이지의 자료만 저장하면 뒤의 글은 검색할 수 없습니다. 그렇다고 페이지를 끝없이 요청해서도 안 됩니다. 정해진 범위 안에서 페이지를 모으고, 나중에 내용을 다시 확인할 수 있는 수집 파일을 만드세요.

사내 게시판을 연결하기 전 연습으로 JSONPlaceholder의 공개 글 API를 사용합니다. 이 서버의 글은 연습용 가짜 내용입니다. 이번에는 문서를 수집하고 저장하는 흐름을 확인하며, 사내 지식을 수집하거나 LLM이 답변을 생성하는 작업은 하지 않습니다.

완성할 결과물은 `collected` 폴더 하나입니다. 여기에 **페이지별 원본 JSON, 추출한 글 JSON, CSV, 수집 로그** 네 파일이 있어야 합니다.

## 수집 범위와 제공 자료

조건은 작성자 `userId=1`, 한 페이지 `PAGE_SIZE=2`건, 최대 `MAX_PAGES=2`페이지입니다. 이 API는 `_page`를 1부터 셉니다. 같은 실행 중에는 작성자와 페이지 크기를 바꾸지 않습니다.

다음 페이지가 있으면 응답의 `Link` 헤더에 `next` 관계가 들어옵니다. 제공 코드가 HTTPX의 `response.links`를 읽어 `has_next` 불리언으로 바꿉니다. `False`면 해당 조건의 마지막 페이지이고, `True`인데 우리가 정한 최대 페이지에 도달했다면 일부만 수집한 것입니다. 이 API에서 페이지 크기만큼 받았다는 사실만으로 마지막 페이지 여부를 판정하지 마세요.

`MODE="live"`는 공개 서버에 실제 GET 요청을 보냅니다. 공개 API 요청에는 timeout이 있으며, 요청이나 페이지 JSON 구조 확인에서 오류가 나면 이미 받은 일부 자료를 버리고 제공 샘플로 처음부터 다시 시작합니다. 이때 결과 모드는 `sample_fallback`입니다. `MODE="sample"`은 외부 호출 없이 샘플만 처리하므로 코드를 수정하며 비교하기에 알맞습니다.

샘플은 글 세 건입니다. ID 순서는 101, 101, 102이고 앞의 두 글은 중복입니다. 마지막 글의 제목은 공백뿐입니다. 첫 글의 본문에는 쉼표와 줄바꿈도 들어 있습니다. **이 문제에서는 이 내용을 정제하거나 제거하지 않고 보관**합니다. 샘플은 2페이지 첫 요청에만 상태 429와 `Retry-After: 1`을 돌려줍니다. 다시 같은 페이지를 요청하면 마지막 한 건을 받습니다.

## TODO 1 — 제한 응답 후 같은 페이지 한 번 다시 받기

`fetch_page(client, page)`의 첫 GET 요청은 제공되어 있습니다. `response`가 그 응답이며, `params`에는 작성자·페이지·페이지 크기가 들어 있습니다. 다음 처리를 채우세요.

- 상태가 429일 때만 `Retry-After` 값을 정수로 읽습니다. 헤더가 없으면 문자열 `1`을 사용합니다.
- 이 실습에서 허용하는 대기는 0~3초입니다. 이 범위를 벗어나면 `ValueError`로 멈춥니다. 정수로 바꿀 수 없는 값도 변환 단계에서 예외가 나면 됩니다.
- 어느 페이지를 몇 초 후 다시 요청하는지 출력하고, `time.sleep`으로 기다립니다.
- **같은 endpoint와 같은 params**로 GET 요청을 한 번 더 보내 `response`를 갱신합니다.
- 429 처리 여부와 관계없이 마지막 응답의 상태를 검사합니다. 다시 429가 와도 세 번째 요청은 보내지 않습니다.

상태 검사는 `.raise_for_status()`를 사용합니다. 이후 JSON 목록을 읽고 `(그 페이지의 글 리스트, has_next)`를 반환하는 코드는 제공됩니다. 실제 API의 Retry-After가 항상 짧은 정수라는 뜻은 아닙니다. 이 프로그램은 지원하지 않는 값이면 멈추고 샘플로 전환하도록 범위를 정했습니다.

## TODO 2 — 성공한 페이지를 모으고 종료하기

`collect(client)`에는 빈 리스트 `pages`, `items`와 초기 `has_next`가 제공됩니다. 결과는 `(pages, items, has_next)` 세 값입니다.

1부터 `MAX_PAGES`까지 반복하며 `fetch_page`를 호출하세요. 성공하면 페이지 번호를 `page`, 반환된 글 리스트를 `items` 키에 담은 딕셔너리를 `pages`에 추가합니다. 전체 글 목록인 함수 안의 `items`에는 각 글을 이어 붙입니다. 여기서 페이지 기록 안의 키 이름 `items`와 전체 목록 변수 `items`는 이름은 같지만 담는 범위가 다릅니다.

다음 페이지가 없거나 최대 페이지까지 받았으면 반복을 끝냅니다. 더 요청할 때만 1초 기다립니다. 함수가 끝날 때 마지막 응답에서 받은 `has_next`를 그대로 반환해야 전체 수집인지 범위 제한에 따른 종료인지 구분할 수 있습니다.

리스트의 `.append()`는 항목 하나를 추가하고, `.extend()`는 다른 리스트의 항목들을 이어 붙입니다. 페이지 기록은 하나씩 추가하고, 글은 페이지 안의 각 글을 전체 목록에 합쳐야 합니다. `range`의 끝값은 포함되지 않는다는 점도 확인하세요.

## TODO 3 — 쉼표와 줄바꿈을 보존하는 CSV 저장

수집 후 제공 코드가 각 글에서 `id`, `title`, `body`를 꺼내 `rows`라는 리스트를 만듭니다. 필수 필드가 없으면 파일을 쓰기 전에 멈춥니다. 중복과 공백 제목은 그대로 남습니다. `output`은 이미 생성된 저장 폴더의 `Path` 객체입니다.

`output` 아래 `items.csv`를 쓰기 모드로 열어 다음 순서로 저장하세요.

1. 파일은 UTF-8 인코딩과 `newline=""` 옵션으로 엽니다.
2. `csv.DictWriter`를 만들고 열 순서를 `id`, `title`, `body`로 고정합니다.
3. 헤더를 한 번 쓰고 `rows` 전체를 씁니다.

`with`로 파일을 열면 작업 후 닫는 처리를 맡길 수 있습니다. `DictWriter`의 `.writeheader()`와 `.writerows()`를 사용하며, 본문을 쉼표로 직접 연결하지 않습니다. TODO 3은 함수 밖에 있으므로 `with` 블록 안쪽 코드만 들여쓰기합니다.

## 실행하고 파일 열기

세 TODO를 채운 뒤 `sample`로 먼저 실행해 정해진 결과와 비교하고, `live`로 한 번 실행해 공개 서버의 결과를 확인하세요. 두 모드의 내용과 건수는 같을 필요가 없습니다. 마지막 셀의 수집과 저장까지 실행해야 파일이 만들어집니다.

노트북에서는 현재 작업 폴더 아래, Python 파일로 실행하면 그 파일 옆에 `collected`가 생깁니다. 같은 위치에서 다시 실행하면 네 파일을 갱신하므로 모드를 바꾸기 전에 출력과 로그를 확인해 두세요. JSON 세 파일의 저장 코드는 제공되어 있습니다.

- `raw_response.json`: 페이지 번호와 각 페이지에서 받은 JSON 목록 전체. 바이트 원문이나 HTTP 헤더 전체를 저장하는 파일은 아닙니다.
- `items.json`: `id`, `title`, `body`만 추출한 글 목록
- `items.csv`: 같은 목록을 고정된 열 순서로 저장한 표
- `collection_log.json`: 실행 모드·요청 조건·건수·완료 여부. 실제 실행에서는 조회를 마친 시각을 UTC로 기록하고, 샘플에서는 `collected_at`을 `null`로 둡니다.

## 실행 준비

**Windows + VS Code + PowerShell + Python 3.12 + uv**

아래 자료 ZIP을 내려받아 압축을 풀고, `pyproject.toml`이 있는 폴더를 VS Code로 엽니다. Python과 Jupyter 확장을 설치하고 **터미널 → 새 터미널**에서 PowerShell을 선택하세요. uv가 없다면 먼저 `python -m pip install uv`를 실행합니다.

```powershell
uv sync --python 3.12
uv run python --version
uv run python solution.py
```

버전 출력이 `Python 3.12.x`인지 확인합니다. 제공된 `.python-version`과 `pyproject.toml`도 Python 3.12를 지정합니다. 해당 Python이 없으면 uv가 준비합니다. 별도의 가상환경 활성화 명령은 필요하지 않습니다.

노트북으로 풀려면 `data_api_chapter03_solution.ipynb`를 열고 오른쪽 위 **커널 선택**에서 이 폴더의 `.venv`를 선택합니다. `uv sync`를 마쳤다면 노트북의 패키지 설치 셀은 건너뛰고 준비 코드부터 실행하세요.

**선택: Google Colab**

Colab을 사용할 때는 **파일 → 노트북 업로드**에서 `data_api_chapter03_solution.ipynb`를 엽니다. 필요한 데이터는 노트북에 포함되어 있습니다. 첫 패키지 설치 셀부터 실행하고, 이미 불러온 패키지의 버전 변경 안내가 나오면 런타임을 다시 시작합니다.

```python
%pip install -q "httpx==0.28.1"
```

## 패키지 설치

Windows에서 `uv sync`를 마쳤다면 이 설치 셀은 건너뜁니다. Colab에서는 설치 셀부터 실행하세요.

In [ ]:
%pip install -q "httpx==0.28.1"


## 1. 수집 범위와 제공 샘플

In [ ]:
import csv
import json
import time
from datetime import datetime, timezone
from pathlib import Path
import httpx

MODE = "live"  # "sample"은 외부 호출 없이 429 상황까지 재현합니다.
BASE_URL = "https://jsonplaceholder.typicode.com"
PAGE_SIZE = 2
MAX_PAGES = 2
SAMPLE = [
    {"id": 101, "title": "교육 신청", "body": "포털에서 신청, 승인 후 수강\n문의는 교육팀"},
    {"id": 101, "title": "교육 신청", "body": "포털에서 신청, 승인 후 수강\n문의는 교육팀"},
    {"id": 102, "title": "   ", "body": "제목이 비어 있는 원본도 그대로 보관합니다."},
]
attempts = {}


def sample_response(request):
    page = int(request.url.params["_page"])
    attempts[page] = attempts.get(page, 0) + 1
    if page == 2 and attempts[page] == 1:
        return httpx.Response(429, headers={"Retry-After": "1"})
    start = (page - 1) * PAGE_SIZE
    next_link = f'<{BASE_URL}/posts?userId=1&_page={page + 1}&_limit={PAGE_SIZE}>; rel="next"'
    headers = {"Link": next_link} if start + PAGE_SIZE < len(SAMPLE) else {}
    return httpx.Response(200, json=SAMPLE[start:start + PAGE_SIZE], headers=headers)




## 2. 같은 조건으로 페이지를 모으기

In [ ]:
def fetch_page(client, page):
    params = {"userId": 1, "_page": page, "_limit": PAGE_SIZE}
    response = client.get("/posts", params=params)
    # TODO 1: 429이면 제한된 시간만 기다리고 같은 페이지를 한 번 재요청합니다.
    if response.status_code == 429:
        seconds = int(response.headers.get("Retry-After", "1"))
        if not 0 <= seconds <= 3:
            raise ValueError("대기 범위를 벗어나 수집을 중단합니다.")
        print("같은 페이지 재요청:", page, "| 대기:", seconds)
        time.sleep(seconds)
        # 실패한 2페이지를 다시 받습니다. 이 요청이 성공하기 전에는 목록에 더하지 않습니다.
        response = client.get("/posts", params=params)
    # 두 번째도 실패하면 예외를 전달하므로 반복해서 기다리지 않습니다.
    response.raise_for_status()
    # TODO 1 끝
    items = response.json()
    if not isinstance(items, list) or any(not isinstance(row, dict) for row in items):
        raise ValueError("글 목록 구조가 아닙니다.")
    return items, "next" in response.links


def collect(client):
    pages, items = [], []
    has_next = False
    # TODO 2: 성공한 페이지만 원본과 전체 목록에 추가합니다.
    for page in range(1, MAX_PAGES + 1):
        batch, has_next = fetch_page(client, page)
        pages.append({"page": page, "items": batch})
        # append(batch)는 중첩 목록이 됩니다. 원본 페이지와 평탄한 전체 목록을 따로 보관합니다.
        items.extend(batch)
        if not has_next or page == MAX_PAGES:
            break
        time.sleep(1)
    # 상한에서 멈춰도 next 신호를 남겨야 complete와 limited를 구분할 수 있습니다.
    return pages, items, has_next
    # TODO 2 끝




## 3. 실행 모드를 결정하고 네 파일 저장하기

In [ ]:
if MODE not in {"live", "sample"}:
    raise ValueError("MODE는 live 또는 sample이어야 합니다.")
used_mode = MODE
transport = httpx.MockTransport(sample_response) if MODE == "sample" else None
try:
    with httpx.Client(base_url=BASE_URL, transport=transport, timeout=5.0) as client:
        pages, items, has_next = collect(client)
except (httpx.HTTPError, ValueError) as error:
    print("수집 실패, 전체를 제공 샘플로 다시 시작:", type(error).__name__)
    used_mode = "sample_fallback"
    attempts.clear()
    with httpx.Client(base_url=BASE_URL, transport=httpx.MockTransport(sample_response)) as client:
        pages, items, has_next = collect(client)

# 필수 필드가 없으면 파일을 쓰기 전에 멈춥니다. 중복과 빈 제목은 유지합니다.
rows = [{key: item[key] for key in ("id", "title", "body")} for item in items]
base = Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()
output = base / "collected"
output.mkdir(exist_ok=True)
collected_at = datetime.now(timezone.utc).isoformat() if used_mode == "live" else None
log = {"mode": used_mode, "source": BASE_URL + "/posts", "userId": 1,
       "collected_at": collected_at, "item_count": len(rows), "page_count": len(pages),
       "status": "limited" if has_next else "complete", "page_size": PAGE_SIZE}
for name, data in [("raw_response.json", pages), ("items.json", rows), ("collection_log.json", log)]:
    (output / name).write_text(json.dumps(data, ensure_ascii=False, indent=2, sort_keys=True), encoding="utf-8")

# TODO 3: 열 순서를 고정하고, 줄바꿈이 있는 본문도 CSV 도구로 저장합니다.
# CSV 도구가 본문의 쉼표·줄바꿈을 인용 처리하며, newline=""는 추가 줄바꿈 변환을 막습니다.
with (output / "items.csv").open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=["id", "title", "body"])
    writer.writeheader()
    writer.writerows(rows)
# TODO 3 끝
print("실행 모드:", used_mode, "| 글 수:", len(rows), "| 수집 범위:", log["status"])
print("ID 순서:", [row["id"] for row in rows])
print("저장 폴더:", output)


## 결과 확인

`sample`의 결과는 글 세 건, ID 순서 `[101, 101, 102]`, 상태 `complete`입니다. 2페이지를 1초 후 재요청한다는 메시지가 한 번 나오고, 저장된 페이지 기록은 두 개여야 합니다. 실패했던 429 응답을 성공 페이지처럼 더하면 안 됩니다.

CSV를 열어 열 순서가 `id`, `title`, `body`인지 확인하세요. 본문의 쉼표와 줄바꿈이 한 글의 `body` 안에 남아 있어야 합니다. 줄바꿈이 있는 필드 때문에 CSV 파일의 눈에 보이는 줄 수는 글 수와 다를 수 있습니다. JSON 목록에서 중복 글 두 건과 마지막 제목의 공백도 확인합니다.

`live`는 실제 서버의 상태에 따라 결과가 달라질 수 있습니다. `limited`라면 최대 두 페이지까지만 저장했고 뒤에 자료가 더 있다는 뜻입니다. `sample_fallback`이면 원본 JSON부터 로그까지 모두 샘플 결과여야 합니다. 로그의 `source` 주소만 보고 실제 수집 성공으로 판단하지 마세요.

## 짧은 관찰 메모

1. 샘플에서 2페이지가 429일 때 바로 다음 페이지로 넘어가면 어떤 글이 빠지나요? 같은 페이지를 다시 요청해도 중복해서 추가되지 않는 이유를 적으세요.
2. `complete`와 `limited`는 무엇이 다른가요? 이번 live 실행의 모드와 상태를 보고 실제로 저장한 범위를 설명하세요.
3. 중복과 공백 제목을 원본에서 지우지 않은 이유를 적으세요. 실제 실행과 샘플 실행의 `collected_at` 차이, 그리고 CSV 본문의 쉼표를 직접 처리하지 않은 이유도 함께 설명하세요.

이 장은 약 60분 동안 TODO 구현과 결과 확인, 관찰 메모 작성까지 진행합니다. 완성한 코드 또는 노트북과 저장한 원본 JSON·항목 JSON·CSV·수집 로그를 제출하세요. 샘플의 요청 페이지가 1→2→2인데 저장 페이지는 두 개인 이유를 적고, 사용한 실행 모드와 `complete`·`limited`에 따른 수집 범위를 로그로 설명합니다.

## 해설

## 429에서 페이지를 유지하는 이유

429는 요청한 페이지를 성공적으로 받았다는 응답이 아닙니다. 샘플의 2페이지는 첫 요청에서 제한 응답을 보내므로, 여기서 페이지 번호를 올리면 ID 102 글을 받지 못합니다. 정답은 기존 `params`를 다시 사용해 같은 2페이지를 한 번 더 요청합니다.

상태 검사는 재요청 처리 뒤에 있습니다. 첫 응답에서 곧바로 예외를 발생시키면 기다렸다가 다시 받는 처리까지 갈 수 없습니다. 반대로 상태 검사를 429 분기 안에만 넣으면 404나 500 같은 다른 실패를 놓칩니다. 마지막 응답을 공통으로 검사해야 두 경우를 모두 다룹니다. 두 번째 응답도 429라면 예외가 발생하고 수집을 멈춥니다.

정수 초로 해석할 수 없는 Retry-After나 0~3초 밖의 값도 실패로 처리합니다. 모든 API의 재시도 규칙을 구현한 것은 아닙니다. 짧은 실습에서 다룰 수 있는 대기와 요청 횟수를 정한 것입니다. 상태가 정상인 페이지 사이에는 별도로 1초 간격을 둡니다.

## 페이지 기록과 글 목록을 나누는 이유

`fetch_page`가 성공한 뒤에만 페이지와 글을 추가합니다. 제한 응답은 이 함수 안에서 처리되고, 최종 실패하면 예외가 올라오므로 실패한 페이지를 목록에 붙이지 않습니다. 샘플의 요청 순서는 1, 2, 2이지만 저장된 페이지 번호는 1, 2입니다.

`pages.append`에는 페이지 번호와 해당 페이지 글 목록을 담은 딕셔너리 하나를 넣습니다. 전체 목록에는 `items.extend`로 글들을 붙입니다. 여기에 `append`를 사용하면 글 목록 안에 또 페이지별 리스트가 들어가 다음 단계가 기대하는 평평한 글 목록이 되지 않습니다.

마지막 응답에 `next`가 없으면 `complete`입니다. 이는 현재 작성자 조건에 맞는 페이지를 끝까지 받았다는 뜻이며 서버 전체의 글을 수집했다는 뜻은 아닙니다. 다음 페이지가 있는데 최대 페이지까지 받았다면 `limited`입니다. 최대 범위에 도달했다고 `has_next`를 거짓으로 바꾸면 이 차이가 사라집니다.

실제 검증에서는 두 페이지에서 네 글을 받고 `limited`가 나왔습니다. 이 수치는 실행 당시의 관찰이며 학생이 다른 시점의 서버 응답을 같은 ID나 제목으로 맞출 필요는 없습니다. 제공 샘플은 두 페이지에 세 건이므로 `complete`가 나옵니다.

## 전체를 샘플로 다시 시작하는 이유

실제 1페이지를 받은 뒤 2페이지에서 실패할 수도 있습니다. 이때 샘플 2페이지를 이어 붙이면 출처가 다른 자료가 하나의 수집 결과가 됩니다. 제공 코드는 `collect`를 새로 호출해 빈 목록부터 만들고 결과를 통째로 교체합니다. 샘플의 재시도 횟수도 초기화합니다.

따라서 `sample_fallback`은 실제 수집 성공의 다른 이름이 아닙니다. 공개 서버의 수집을 완료하지 못해 제공 자료로 저장 동작을 확인했다는 표시입니다. 실제 실패 이전의 일부 자료는 최종 네 파일에 남기지 않습니다. JSON 목록 구조에 문제가 있으면 수집을 다시 시작하고, 글에 필수 필드가 없다면 저장 전 추출 단계에서 멈추게 되어 있습니다.

## 원본과 CSV를 보관하는 방법

원본 JSON에는 각 페이지에서 받은 글의 필드를 그대로 남깁니다. 추출 JSON과 CSV에는 이번에 필요한 세 필드만 넣습니다. 둘을 나누면 나중에 추출 기준을 바꿀 때 원래 응답을 다시 확인할 수 있습니다. 여기서 원본은 해석된 JSON 구조를 뜻하며 응답 바이트를 그대로 저장한 것은 아닙니다.

중복 글과 공백 제목을 지금 제거하면 받은 내용과 정제한 내용을 구분하기 어려워집니다. 이 문제는 수집과 저장을 다루므로 세 건을 모두 남깁니다. JSON은 UTF-8과 `ensure_ascii=False`로 한글을 읽기 쉽게 쓰고, 키 순서를 고정해 같은 샘플을 비교하기 쉽게 합니다.

CSV의 본문에는 쉼표와 줄바꿈이 들어 있습니다. 직접 쉼표로 이어 붙이면 본문 일부가 다른 열이나 다른 행처럼 읽힐 수 있습니다. `DictWriter`는 필요한 인용 처리를 해 줍니다. `newline=""`은 CSV 도구가 줄바꿈을 처리하도록 하는 파일 옵션입니다. CSV를 다시 읽었을 때 필드 내용이 유지되는지가 중요하며, 텍스트 편집기에 표시되는 줄 수와 글 수를 같게 만들 필요는 없습니다.

## 관찰 메모 예시

첫 메모에는 “2페이지를 건너뛰면 ID 102가 빠진다. 성공 응답을 받은 다음에만 목록에 붙이므로 재요청했다는 이유로 같은 페이지가 두 번 저장되지는 않는다”고 쓸 수 있습니다. 원래 샘플에 들어 있는 ID 101 중복은 별개의 문제입니다.

두 번째 메모는 실제 출력에 맞춰 적습니다. 예를 들어 “live와 limited가 나왔다. 작성자 1의 첫 두 페이지, 네 건을 저장했으며 뒤의 글은 받지 않았다”는 설명입니다. fallback이었다면 “제공 샘플 세 건을 저장했고 실제 서버의 범위는 확인하지 못했다”고 적습니다.

마지막 메모에는 “원본과 정제 결과를 구분하기 위해 중복과 공백을 보관했다. 샘플은 실제 수집 시각이 없으므로 null이고, 실제 실행의 시각도 글의 작성일과는 다르다. CSV 도구에 인용 처리를 맡겨 본문 안의 쉼표와 줄바꿈을 보존했다”는 내용이 들어가면 됩니다.